# Sugarcane cultivar identification with YOLOv8n-cls: grouped partition, reference run (S1 File)

This notebook reproduces **one complete run** (the reference run, seed 101) of the pipeline described in the paper *"Sugarcane cultivar identification under controlled conditions: a deep learning classification approach"* (PLOS ONE). It replaces the earlier version of this file, which used an image-level random partition.

**What it does**

1. Reads the table of near-duplicate groups (`dimensoes_grupos.csv`) and checks it against the list of near-duplicate pairs (`duplicatas.csv`), both produced by `looking_for_duplicates.ipynb`.
2. Reports the pixel values of the background in the cropped images (diagnostic).
3. Builds the grouped 70/15/15 partition for the chosen seed, with each group in a single subset, and verifies that no group is split.
4. Fine-tunes the pre-trained YOLOv8n-cls (30 epochs, patience 10, input size 224).
5. Evaluates the saved checkpoint on the held-out test subset: accuracy with Wilson interval, confusion matrix, class-wise precision, recall and F1-score, misclassified images.
6. Draws the learning curve, measures single-image inference time (batch size 1), and draws Grad-CAM maps.
7. Records the software environment.

**What it does not do.** The 10 runs of the paper, the color-only baselines (SVM and Random Forest), and the post-hoc analyses of all runs are in the notebooks of S2 File. A single run on different hardware can differ by one or two test images from the values reported in the paper, because GPU kernels are not fully deterministic.

**Input files**

| File | Content |
|---|---|
| `data_cropped/<cultivar>/*.JPG` | 1,476 cropped images, one folder per cultivar (CTC_1007, IAC_5503, RB_5375, RB_7515) |
| `data_cropped/inspecao/dimensoes_grupos.csv` | one row per image, with the group it belongs to (column `grupo`) |
| `data_cropped/inspecao/duplicatas.csv` | pairs of near-duplicate images (same orientation, aspect ratios within 5%, Pearson correlation > 0.95 between grayscale images resized to 256 x 64 pixels; columns `a`, `b`, `corr`), produced by `looking_for_duplicates.ipynb` |

**Run time.** About 15 to 30 minutes on a GPU (Google Colab T4). A GPU is needed for training.

In [ ]:
# Install the Ultralytics version used for the results in the paper (falls back to the latest if unavailable).
!pip install -q ultralytics==8.4.208 || pip install -q ultralytics

In [ ]:
import os, sys, json, math, random, shutil, glob, platform
import numpy as np, pandas as pd

# ======================= CONFIGURATION =======================
BASE       = '/content/drive/MyDrive/Colab_Notebooks/four_varieties'
DATA_DIR   = f'{BASE}/data_cropped'                              # cropped images, one sub-folder per cultivar
GROUPS_CSV = f'{BASE}/data_cropped/inspecao/dimensoes_grupos.csv'  # group of each image
PAIRS_CSV  = f'{BASE}/data_cropped/inspecao/duplicatas.csv'        # near-duplicate pairs (correlation > 0.95)
WORK_DIR   = f'{BASE}/runs_agrupado/S1_reference_run'              # all outputs are written here
CLASSES    = ['CTC_1007', 'IAC_5503', 'RB_5375', 'RB_7515']        # data_cropped also holds other sub-folders (e.g. inspecao), which are ignored

SEED     = 101                  # reference run of the paper
EPOCHS   = 30
PATIENCE = 10
IMGSZ    = 224
SPLIT    = (0.70, 0.15, 0.15)   # train / validation / test, applied per cultivar
IMG_EXT  = ('.jpg', '.jpeg', '.png')
GRADCAM_PER_CLASS = 3
# =============================================================

if 'google.colab' in sys.modules or os.path.exists('/content'):
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except Exception as e:
        print('Drive was not mounted:', e)

os.makedirs(WORK_DIR, exist_ok=True)
assert os.path.isdir(DATA_DIR), f'Folder not found: {DATA_DIR}'
for c in CLASSES: assert os.path.isdir(os.path.join(DATA_DIR, c)), f'Cultivar folder not found: {c}'
ALL = {c: sorted(f for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(IMG_EXT)) for c in CLASSES}
print('Cultivars:', CLASSES)
print('Images per cultivar:', {c: len(v) for c, v in ALL.items()}, '| total:', sum(len(v) for v in ALL.values()))

## 1. Near-duplicate groups

The same photograph, or photographs that differ only marginally, appear in the data set under different file identifiers. Two images are linked when they have the same orientation, aspect ratios within 5%, and a Pearson correlation above 0.95 between their grayscale versions resized to 256 x 64 pixels (computed in `looking_for_duplicates.ipynb`, which also writes the two CSV files read here). Images linked directly or through other images form one group (connected components). Each group is later kept in a single subset.

The next cell reads the group table and checks that its groups are exactly the connected components of the pairs listed in `duplicatas.csv`.

In [ ]:
G = pd.read_csv(GROUPS_CSV)
G['name'] = G['file'].map(os.path.basename)
GROUP = {(r.classe, r.name): r.grupo for r in G.itertuples()}

on_disk = {(c, f) for c in CLASSES for f in ALL[c]}
assert on_disk == set(GROUP), 'The files in DATA_DIR and in the group table differ.'

# connected components of the graph of near-duplicate pairs
P = pd.read_csv(PAIRS_CSV)
parent = {f: f for f in G['file']}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in zip(P['a'], P['b']):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
comp = G['file'].map(find)
same = (comp.groupby(G['grupo']).nunique() == 1).all() and (G['grupo'].groupby(comp).nunique() == 1).all()
assert same, 'The groups in the table are not the connected components of the listed pairs.'

sizes = G.groupby('grupo').size()
print(f'Pairs listed: {len(P)} (minimum correlation {P["corr"].min():.4f})')
print(f'Groups: {len(sizes)} | images in groups of two or more: {int(sizes[sizes >= 2].sum())} ({sizes[sizes >= 2].sum() / len(G) * 100:.1f}%) | largest group: {int(sizes.max())} images')
print('Check passed: the groups are the connected components of the pairs.')

## 2. Diagnostic: background pixels in the cropped images

The cropped images are bounding-box crops that keep the dark studio base around the stalk. The cell below measures, for every image, the share of pixels whose maximum channel value is above 5, the share at or below 25, and the mean value of the four 8 x 8 corners. In the paper, this threshold (above 5) is the one used by the color baselines, and it retains almost the whole crop.

In [ ]:
from PIL import Image
rows = []
for c in CLASSES:
    for f in ALL[c]:
        a = np.asarray(Image.open(os.path.join(DATA_DIR, c, f)).convert('RGB'))
        mx = a.max(2)
        corners = np.concatenate([a[:8, :8].reshape(-1, 3), a[:8, -8:].reshape(-1, 3), a[-8:, :8].reshape(-1, 3), a[-8:, -8:].reshape(-1, 3)])
        rows.append(dict(classe=c, arquivo=f, h=a.shape[0], w=a.shape[1], frac_max_gt5=(mx > 5).mean(), frac_max_le25=(mx <= 25).mean(),
                         corner_mean=corners.mean(), frac_exact_zero=(mx == 0).mean()))
BG = pd.DataFrame(rows); BG.to_csv(os.path.join(WORK_DIR, 'background_diagnostic.csv'), index=False)
print(BG.groupby('classe')[['frac_max_gt5', 'frac_max_le25', 'corner_mean', 'frac_exact_zero']].mean().round(3).to_string())
print('\nAll images: share of pixels with max channel > 5: mean %.3f | exactly zero: mean %.4f' % (BG.frac_max_gt5.mean(), BG.frac_exact_zero.mean()))

## 3. Grouped partition (70/15/15 per cultivar)

For each cultivar, the groups are listed in sorted order and shuffled with the seed; they are allocated to the test subset first and to the validation subset second, without exceeding the target counts; the remaining groups form the training subset. The targets are `floor(0.70 N)` for training and `floor(0.15 N)` for validation, and the test subset receives the rest. With the four cultivars this gives 1,031 training, 220 validation, and 225 test images in every run.

In [ ]:
def split_grouped(seed):
    rng = random.Random(seed)
    out = {'train': [], 'val': [], 'test': []}
    for c in CLASSES:
        files = ALL[c]; n = len(files)
        n_tr, n_va = int(n * SPLIT[0]), int(n * SPLIT[1]); n_te = n - n_tr - n_va
        groups = {}
        for f in files: groups.setdefault(GROUP[(c, f)], []).append(f)
        order = sorted(groups); rng.shuffle(order)
        dest = {'test': [], 'val': [], 'train': []}; free = order
        for part, target in (('test', n_te), ('val', n_va)):
            count, rest = 0, []
            for g in free:
                if count + len(groups[g]) <= target: dest[part] += groups[g]; count += len(groups[g])
                else: rest.append(g)
            free = rest
        for g in free: dest['train'] += groups[g]
        for part in out: out[part] += [(c, f) for f in sorted(dest[part])]
    return out

def groups_split_across_subsets(sp):
    seen = {}
    for part, lst in sp.items():
        for c, f in lst: seen.setdefault(GROUP[(c, f)], set()).add(part)
    return sum(1 for s in seen.values() if len(s) > 1)

SP = split_grouped(SEED)
sizes_ = {p: len(v) for p, v in SP.items()}
print('Subset sizes:', sizes_)
assert sum(sizes_.values()) == len(on_disk)
assert groups_split_across_subsets(SP) == 0, 'A group is split across subsets.'
print('Check passed: no group is split across subsets.')
print('Test images per cultivar:', pd.Series([c for c, f in SP['test']]).value_counts().sort_index().to_dict())
pd.DataFrame([(c, f, GROUP[(c, f)], p) for p in SP for c, f in SP[p]], columns=['classe', 'arquivo', 'grupo', 'particao']
             ).to_csv(os.path.join(WORK_DIR, f'split_grouped_seed_{SEED}.csv'), index=False)

## 4. Training

YOLOv8n-cls pre-trained on ImageNet-1K, fine-tuned end to end (no frozen layers) with the Ultralytics defaults, except for the settings below: 30 epochs, early stopping with patience 10 on the validation Top-1 accuracy, input size 224 and the seed of the run. With `optimizer = auto`, Ultralytics selects AdamW with learning rate 0.00125 and weight decay 0.0005 for this number of iterations; the batch size is the default (16). Training uses the Ultralytics classification augmentations (horizontal flip, random resized crop, HSV jitter, RandAugment, random erasing); validation and test images are resized (shorter side to 224 pixels) and center-cropped to 224 x 224.

In [ ]:
from ultralytics import YOLO
import ultralytics, torch
print('Ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (training on CPU will be very slow)')

RUN_DIR = os.path.join(WORK_DIR, f'run_seed_{SEED}')
DS = os.path.join(RUN_DIR, 'dataset_split')
BEST = os.path.join(RUN_DIR, 'train', 'weights', 'best.pt')

if not os.path.exists(BEST):
    shutil.rmtree(DS, ignore_errors=True)
    for part, lst in SP.items():
        for c, f in lst:
            d = os.path.join(DS, part, c); os.makedirs(d, exist_ok=True)
            shutil.copy(os.path.join(DATA_DIR, c, f), os.path.join(d, f))
    model = YOLO('yolov8n-cls.pt')
    model.train(data=DS, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, seed=SEED, project=RUN_DIR, name='train', exist_ok=True, verbose=False)
else:
    print('best.pt already exists; training skipped:', BEST)

log = pd.read_csv(os.path.join(RUN_DIR, 'train', 'results.csv')); log.columns = [c.strip() for c in log.columns]
acc = log['metrics/accuracy_top1'].values
stop_epoch = len(log)
first_max = int(acc.argmax()) + 1                          # first epoch with the maximum validation Top-1 accuracy
last_tie  = int(np.where(acc == acc.max())[0].max()) + 1   # last epoch tying the maximum: the one saved as best.pt
print(f'Executed epochs: {stop_epoch} | maximum validation Top-1: {acc.max():.4f} | first maximum: epoch {first_max} | best.pt: epoch {last_tie} | epochs at the maximum: {int((acc == acc.max()).sum())}')

## 5. Test evaluation, confusion matrix, class-wise metrics, and inference time

Each test image is classified one at a time (batch size 1) after 10 warm-up images. The times come from the Ultralytics profiler; preprocessing includes reading and decoding the JPEG file and resizing it, because predictions are made from the file paths.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import matplotlib; import matplotlib.pyplot as plt

def wilson(k, n, z=1.959964):
    p = k / n; den = 1 + z * z / n; ctr = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return ctr - half, ctr + half

mdl = YOLO(BEST)
NAMES = [mdl.names[i] for i in range(len(mdl.names))]
assert NAMES == CLASSES, f'Class order of the model {NAMES} differs from {CLASSES}'
test = sorted(SP['test'])
paths = [os.path.join(DATA_DIR, c, f) for c, f in test]
for p in paths[:10]: mdl.predict(p, imgsz=IMGSZ, verbose=False)          # warm-up
pre, inf, post, pred, prob = [], [], [], [], []
for p in paths:
    r = mdl.predict(p, imgsz=IMGSZ, verbose=False)[0]
    pred.append(int(r.probs.top1)); prob.append(float(r.probs.top1conf))
    pre.append(r.speed['preprocess']); inf.append(r.speed['inference']); post.append(r.speed['postprocess'])
real = np.array([CLASSES.index(c) for c, f in test]); pred = np.array(pred)
correct = int((real == pred).sum()); n = len(real)
lo, hi = wilson(correct, n)
print(f'Test Top-1 accuracy: {correct}/{n} = {correct / n:.4f} | 95% Wilson interval [{lo:.4f}, {hi:.4f}]')

cm = pd.crosstab(pd.Series(real, name='true'), pd.Series(pred, name='predicted')).reindex(index=range(len(CLASSES)), columns=range(len(CLASSES)), fill_value=0)
cm.index = CLASSES; cm.columns = CLASSES
print('\nConfusion matrix (rows: true, columns: predicted)'); print(cm.to_string())
cm.to_csv(os.path.join(WORK_DIR, f'confusion_matrix_seed_{SEED}.csv'))

Pr, Rc, F1, _ = precision_recall_fscore_support(real, pred, labels=range(len(CLASSES)), zero_division=0)
CLS = pd.DataFrame(dict(cultivar=CLASSES + ['macro'], precision=list(Pr) + [Pr.mean()], recall=list(Rc) + [Rc.mean()], f1=list(F1) + [F1.mean()]))
print('\nClass-wise metrics'); print(CLS.round(3).to_string(index=False))
CLS.to_csv(os.path.join(WORK_DIR, f'classwise_metrics_seed_{SEED}.csv'), index=False)

PRED = pd.DataFrame(dict(cultivar=[c for c, f in test], file=[f for c, f in test], true=real, predicted=pred, prob_predicted=prob))
PRED['correct'] = PRED.true == PRED.predicted
PRED.to_csv(os.path.join(WORK_DIR, f'test_predictions_seed_{SEED}.csv'), index=False)
err = PRED[~PRED.correct].assign(true=lambda d: [CLASSES[i] for i in d.true], predicted=lambda d: [CLASSES[i] for i in d.predicted])
print('\nMisclassified test images'); print(err[['file', 'true', 'predicted', 'prob_predicted']].to_string(index=False))

fig, ax = plt.subplots(figsize=(4.6, 4.0)); ax.imshow(cm.values, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=35, ha='right', fontsize=8)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, int(cm.values[i, j]), ha='center', va='center', fontsize=9, color='white' if cm.values[i, j] > cm.values.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, f'confusion_matrix_seed_{SEED}.png'), dpi=300); plt.show()

lat = dict(preprocess_ms=float(np.mean(pre)), inference_ms=float(np.mean(inf)), postprocess_ms=float(np.mean(post)))
lat['total_ms'] = sum(lat.values())
print('\nMean inference time per image (ms):', {k: round(v, 2) for k, v in lat.items()})

## 6. Learning curve

Training loss and validation loss (left axis) and validation Top-1 accuracy (right axis). The red dashed line marks the first epoch with the maximum validation Top-1 accuracy, which starts the patience counter; the green dashed line marks the last epoch that ties this maximum, whose weights are saved as `best.pt`; the shaded interval is the patience window.

In [ ]:
ep = np.arange(1, len(log) + 1); tl, vl = log['train/loss'].values, log['val/loss'].values
BLUE, RED, GREEN = '#1f77b4', '#d62728', '#2ca02c'
fig, ax = plt.subplots(figsize=(10, 5.5)); ax2 = ax.twinx()
ax.axvspan(first_max, stop_epoch + 0.8, color='#999999', alpha=0.13, lw=0, zorder=0)
ax.axvline(first_max, color=RED, ls='--', lw=1.6, alpha=0.85, zorder=1)
ax.axvline(last_tie, color=GREEN, ls='--', lw=1.6, alpha=0.85, zorder=1)
l1, = ax.plot(ep, tl, '-o', color=BLUE, lw=2.6, ms=5, label='Training loss', zorder=3)
l2, = ax.plot(ep, vl, '-s', color=RED, lw=2.6, ms=5, label='Validation loss', zorder=3)
l3, = ax2.plot(ep, acc, '--^', color=GREEN, lw=2.6, ms=6, label='Validation Top-1 accuracy', zorder=4)
ax2.plot([first_max, last_tie], [acc.max()] * 2, 's', color=GREEN, ms=11, zorder=5)
ymax = max(1.25, 1.05 * max(tl.max(), vl.max()))
ax.set_xlim(0.4, stop_epoch + 0.8); ax.set_xticks(range(1, stop_epoch + 1, 2)); ax.set_ylim(-0.02, ymax)
ax2.set_ylim(max(0.0, acc.min() - 0.05), 1.025)
ax.set_xlabel('Epoch', fontsize=14, fontweight='bold'); ax.set_ylabel('Cross-entropy loss', fontsize=14, fontweight='bold')
ax2.set_ylabel('Validation Top-1 accuracy', fontsize=14, fontweight='bold')
ax.grid(True, ls=':', alpha=0.45); ax.set_axisbelow(True)
ax.legend(handles=[l1, l2, l3], loc='center', bbox_to_anchor=(0.38, 0.52), fontsize=12, framealpha=0.95).set_zorder(10)
ax.text(first_max + 0.15, ymax * 1.005, f'Patience window ({PATIENCE} epochs)', fontsize=10.5, style='italic', color='#666666', va='bottom', clip_on=False)
ax.text(first_max + 0.25, 0.60, f'Epoch {first_max}: first epoch with\nmaximum validation Top-1\naccuracy ({acc.max():.4f})', fontsize=10.5, va='top')
ax.text(last_tie - 0.25, 0.36, f'Epoch {last_tie}: last epoch tying\nthe maximum; saved as best.pt', fontsize=10.5, va='top', ha='right')
fig.tight_layout(); fig.savefig(os.path.join(WORK_DIR, f'learning_curve_seed_{SEED}.png'), dpi=300, bbox_inches='tight'); plt.show()
log.to_csv(os.path.join(WORK_DIR, f'training_log_seed_{SEED}.csv'), index=False)

## 7. Grad-CAM

Class-activation maps from the last backbone stage (module 8, the last C2f block) for the predicted class. They are a qualitative illustration only: the crops keep the dark base around the stalk and no stalk segmentation is applied here, so the share of activation on the stalk is not quantified, and the maps cannot separate morphological from chromatic or surface-reflectance cues. The input shown is the central 224 x 224 square that the network receives at test time.

In [ ]:
from ultralytics.data.augment import classify_transforms
TARGET_LAYER = 8
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = YOLO(BEST).model.float().eval().to(dev)
tf = classify_transforms(size=IMGSZ)

def center_view(rgb, size=IMGSZ):
    h, w = rgb.shape[:2]; s = size / min(h, w); nh, nw = max(size, round(h * s)), max(size, round(w * s))
    im = np.array(Image.fromarray(rgb).resize((nw, nh), Image.BILINEAR)); t, l = (nh - size) // 2, (nw - size) // 2
    return im[t:t + size, l:l + size]

def gradcam(path):
    rgb = np.asarray(Image.open(path).convert('RGB'))
    x = tf(Image.fromarray(rgb)).unsqueeze(0).to(dev).requires_grad_(True); store = {}
    h = net.model[TARGET_LAYER].register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    with torch.enable_grad():
        out = net(x); logits = out[1] if isinstance(out, (tuple, list)) else out
        c = int(logits.argmax(1)); g = torch.autograd.grad(logits[0, c], store['a'])[0]
    h.remove()
    cam = torch.relu((g.mean((2, 3), keepdim=True) * store['a']).sum(1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=(IMGSZ, IMGSZ), mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy()
    return center_view(rgb), (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), c

rng = random.Random(0)
fig, axes = plt.subplots(len(CLASSES), GRADCAM_PER_CLASS * 2, figsize=(3 * GRADCAM_PER_CLASS * 2 * 0.9, 3 * len(CLASSES) * 0.9), squeeze=False)
for r, cn in enumerate(CLASSES):
    items = PRED[(PRED.true == r) & PRED.correct].file.tolist(); rng.shuffle(items)
    for k, f in enumerate(items[:GRADCAM_PER_CLASS]):
        im, cam, _ = gradcam(os.path.join(DATA_DIR, cn, f))
        a0, a1 = axes[r, 2 * k], axes[r, 2 * k + 1]
        a0.imshow(im); a1.imshow(im); a1.imshow(cam, cmap='jet', alpha=0.45)
        a0.set_title(cn if k == 0 else '', fontsize=9); a0.axis('off'); a1.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, f'gradcam_seed_{SEED}.png'), dpi=300); plt.show()

E = PRED[~PRED.correct].head(12)
if len(E):
    fig, axes = plt.subplots(len(E), 2, figsize=(4.2, 2.1 * len(E)), squeeze=False)
    for i, r in enumerate(E.itertuples()):
        im, cam, _ = gradcam(os.path.join(DATA_DIR, r.cultivar, r.file))
        axes[i, 0].imshow(im); axes[i, 1].imshow(im); axes[i, 1].imshow(cam, cmap='jet', alpha=0.45)
        axes[i, 0].set_title(f'true {r.cultivar}\npredicted {CLASSES[r.predicted]}', fontsize=7); axes[i, 0].axis('off'); axes[i, 1].axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, f'gradcam_errors_seed_{SEED}.png'), dpi=300); plt.show()

## 8. Software environment and summary

In [ ]:
import cv2, sklearn, matplotlib as mpl
env = dict(python=platform.python_version(), ultralytics=ultralytics.__version__, torch=torch.__version__, opencv=cv2.__version__,
           numpy=np.__version__, pandas=pd.__version__, scikit_learn=sklearn.__version__, matplotlib=mpl.__version__,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
summary = dict(seed=SEED, subset_sizes=sizes_, stop_epoch=stop_epoch, first_max_epoch=first_max, best_pt_epoch=last_tie,
               test_correct=correct, test_n=n, test_top1=correct / n, wilson95=[lo, hi], inference_ms=lat, environment=env)
json.dump(summary, open(os.path.join(WORK_DIR, f'summary_seed_{SEED}.json'), 'w'), indent=2)
print(json.dumps(summary, indent=2))
print('\nValues reported in the paper for the reference run (seed 101): 27 executed epochs, first maximum at epoch 17, best.pt at epoch 27, test accuracy 223/225 = 0.9911, Wilson interval [0.9682, 0.9976].')
print('Small differences are possible on other hardware or library versions.')